# S01 — dated settlements (post offices + Census urban places)
Task `tasks/T-016-s01-settlements.md` (brief B-030; source choice `research/R9-settlements.md`). Exact-match joins only, no ML/LLM, no web geocoder. Reads the post-office file **remotely** (nothing saved to `data/raw/`, change of 2026-11-24) `research/urban_places_gibson.csv` and `research/urban_coords_override.csv`; writes only `web/data/settlements.json`.

**Sources**
- Post offices: Blevins and Helbock (2021), *US Post Offices*, Harvard Dataverse V1, https://doi.org/10.7910/DVN/NUKCNA. Licence **CC0 1.0**. File `us-post-offices.tab` (id 4491713, 32.4 MB, comma-separated despite `.tab`), read from `https://dataverse.harvard.edu/api/access/datafile/4491713?format=original`. **Fetched 2026-11-24** (read into memory with DuckDB, not stored). Coordinates come from a GNIS match by name + county + state; `GNIS.Dist` (0.75–1) is the match score.
- Urban places: Gibson (1998), Census Population Division Working Paper 27, tables 2–7 (1790–1840), US federal work, public domain → `research/urban_places_gibson.csv` (T-016 step 3).
- Hand coordinates: `research/urban_coords_override.csv` (R9 §5, 2026-11-24): USGS GNIS Populated Place points, two landmark points marked unverified.
- Extent: `web/data/relief_bounds.json` `extent` (shared map frame, T-011).

**Grain of the output** — `settlements.json`: one row per post-office record (`kind = post_office`; a closure of 10+ years or a rename starts a new record) and one row per Gibson urban place (`kind = urban_2500`).

In [1]:
import sys, json, re
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path: sys.path.insert(0, str(PIPELINE_DIR))
import config
import duckdb, numpy as np, pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 200)

WEB = config.PROJECT_DIR / 'web' / 'data'
GIBSON_CSV = config.PROJECT_DIR / 'research' / 'urban_places_gibson.csv'
OVERRIDE_CSV = config.PROJECT_DIR / 'research' / 'urban_coords_override.csv'
UNVERIFIED = {('Moyamensing', 'PA'), ('Southwark', 'PA')}   # landmark points, no GNIS feature (R9 §5)
PO_URL = 'https://dataverse.harvard.edu/api/access/datafile/4491713?format=original'
PO_FETCHED = '2026-11-24'
YEAR_MAX = 1840
YEAR_MIN_VALID = 1600       # 3-digit Established values (185, 190 ...) are truncated years, dropped
CENSUS = [1790, 1800, 1810, 1820, 1830, 1840]
GIBSON_N = {1790: 24, 1800: 33, 1810: 46, 1820: 61, 1830: 90, 1840: 100}   # places per table header (step 3)
EXTENT = json.loads((WEB / 'relief_bounds.json').read_text())['extent']   # [west, south, east, north]
W, S, E, N = EXTENT
con = duckdb.connect()
def q(sql): return con.sql(sql).df()
print('extent:', np.round(EXTENT, 4))

extent: [-100.       24.9083  -66.6167   47.4417]


## 1. Post offices (Blevins and Helbock 2021)
Read every column as text, then cast. `Latitude`/`Longitude` are empty where the office was not geocoded (about a third of the file).

In [2]:
con.sql(f"create or replace table po_raw as select * from read_csv('{PO_URL}', header = true, all_varchar = true)")
raw_n = q("select count(*) n, count(distinct ID) ids, count(Latitude) geocoded from po_raw")
display(raw_n)
assert raw_n.n[0] == 166_140 and raw_n.ids[0] == raw_n.n[0], raw_n      # R9: 166,140 rows; ID is the key
assert raw_n.geocoded[0] == 112_521, raw_n

,n,ids,geocoded
0,166140,166140,112521


In [3]:
con.sql(f'''create or replace table po_all as
  select ID::int id, Name name, AltName alt_name, State state, County1 county,
         try_cast(Established as int) year_established, try_cast(Discontinued as int) year_discontinued,
         try_cast(Latitude as double) lat, try_cast(Longitude as double) lon,
         try_cast("GNIS.Dist" as double) gnis_dist, "GNIS.Feature.Class" gnis_class
  from po_raw''')
funnel = q(f'''select count(*) all_rows,
  count(*) filter (where year_established < {YEAR_MIN_VALID}) bad_year,
  count(*) filter (where year_established between {YEAR_MIN_VALID} and {YEAR_MAX}) est_le_1840,
  count(*) filter (where year_established between {YEAR_MIN_VALID} and {YEAR_MAX} and lat is not null) with_coords,
  count(*) filter (where year_established between {YEAR_MIN_VALID} and {YEAR_MAX} and lat between {S} and {N} and lon between {W} and {E}) inside_extent,
  count(*) filter (where year_established is null) undated
  from po_all''')
display(funnel)
con.sql(f'''create or replace table po as select * from po_all
  where year_established between {YEAR_MIN_VALID} and {YEAR_MAX} and lat between {S} and {N} and lon between {W} and {E}''')
display(q('''select count(*) n, min(year_established) first, max(year_established) last,
  count(*) filter (where year_discontinued is null) open_ended,
  count(*) filter (where year_discontinued < year_established) disc_before_est,
  count(*) filter (where gnis_dist is null) no_gnis_dist,
  count(*) filter (where gnis_dist < 1) gnis_dist_lt_1 from po'''))

,all_rows,bad_year,est_le_1840,with_coords,inside_extent,undated
0,166140,17,20008,13470,13469,44


,n,first,last,open_ended,disc_before_est,no_gnis_dist,gnis_dist_lt_1
0,13469,1639,1840,5076,0,444,954


In [4]:
# where the geocoded <= 1840 offices outside the extent are, and the early tail (R9 caveat: thin pre-1790 coverage)
display(q(f'''select state, count(*) n from po_all where year_established between {YEAR_MIN_VALID} and {YEAR_MAX} and lat is not null
  and not (lat between {S} and {N} and lon between {W} and {E}) group by 1 order by 2 desc'''))
display(q(f'''select name, state, county, year_established, year_discontinued from po_all where year_established < {YEAR_MIN_VALID}'''))
display(q('''select (year_established // 10) * 10 as "decade", count(*) n from po where year_established < 1790 group by 1 order by 1'''))

,state,n
0,FL,1


,name,state,county,year_established,year_discontinued
0,GRASS CREEK VILLAGE,IN,FULTON,188,1884
1,BIRD NEST,KS,PAWNEE,188,1891
2,BUFFALO,KS,WILSON,186,<NA>
3,HASSE,TX,COMANCHE,190,<NA>
4,HAYSVILLE,KS,JOHNSON,185,1860
5,LARNED,OK,ROGER MILLS,190,1906
6,LAWN,TX,TAYLOR,189,<NA>
7,LAZBUDDIE,TX,PARMER,192,<NA>
8,MISSLER,KS,MEADE,191,1933
9,NIX,TX,LAMPASAS,188,1906


,decade,n
0,1630,1
1,1690,1
2,1760,1
3,1770,59
4,1780,39


## 2. Urban places ≥ 2,500 (Gibson 1998)
One row per place + state. Two places are merged into one row each (2026-11-24, `merge_note` column): **Sherburne → Nantucket, MA** (1790 name; the town was renamed in 1795) and **Alexandria, VA** (Gibson lists it under VA in 1790 and DC in 1800–1840, the years it lay in the District, 1801–1846).
`year_established` = first census year with ≥ 2,500 people, a proxy, not a founding date.
1840 caveat: Gibson's 1840 table is the top 100 only (≥ 4,226). An 1830 urban place absent from 1840 is **carried** into 1840 (`carried_1840 = true`, population null): it is not known to have dropped below 2,500.

In [5]:
g = pd.read_csv(GIBSON_CSV, dtype={f'pop_{y}': 'Int64' for y in CENSUS})
assert not g.duplicated(['place', 'state']).any()
per_year = {y: int(g[f'pop_{y}'].notna().sum()) for y in CENSUS}
print('places per census:', per_year)
assert per_year == GIBSON_N, per_year                   # the merges keep one value per year
assert g[[f'pop_{y}' for y in CENSUS]].notna().any(axis=1).all()
display(g[g.merge_note.notna()][['place', 'state', 'alt_name'] + [f'pop_{y}' for y in CENSUS] + ['merge_note']])

g['year_established'] = g[[f'pop_{y}' for y in CENSUS]].notna().idxmax(axis=1).str[4:].astype(int)
g['carried_1840'] = g.pop_1830.notna() & g.pop_1840.isna()
print('carried into 1840:', g[g.carried_1840].place.str.cat(g[g.carried_1840].state, ', ').tolist())

places per census: {1790: 24, 1800: 33, 1810: 46, 1820: 61, 1830: 90, 1840: 100}


,place,state,alt_name,pop_1790,pop_1800,pop_1810,pop_1820,pop_1830,pop_1840,merge_note
31,Nantucket,MA,Sherburne,4620,5617,6807,7266,7202,9012,Gibson 1790 lists the town as Sherburne (Nantu...
103,Alexandria,VA,Alexandria DC,2748,4971,7227,8218,8241,8459,Gibson lists Alexandria under VA in 1790 and D...


carried into 1840: ['Middletown, CT', 'Norwich, CT', 'Annapolis, MD', 'Hagerstown, MD', 'Natchez, MS', 'New Bern, NC', 'Elizabeth, NJ', 'Trenton, NJ', 'Chillicothe, OH', 'Fredericksburg, VA']


### 2a. Coordinates by exact name + state
Upper-cased Gibson `place` must equal the post-office `Name` and `State` (if no `Name` hits, `AltName`); the office must be geocoded.
With several hits the office kept is the one open in the most of the place's census years (listed or carried), then the earliest `Established`. Coordinates are the office's GNIS point, a town-level location.
State is the state in Gibson; Wheeling (VA in 1840) is WV in the post-office file, so it does not match by rule.

In [6]:
g['key'] = g.place.str.upper()
g['years'] = [[y for y in CENSUS if pd.notna(r[f'pop_{y}']) or (y == 1840 and r.carried_1840)] for _, r in g.iterrows()]
con.register('g_v', g[['place', 'state', 'key']])
cand = q('''with h as (
    select g.place, g.state, 'Name' matched_on, p.* from g_v g join po_all p on p.state = g.state and p.name = g.key
    union all
    select g.place, g.state, 'AltName', p.* from g_v g join po_all p on p.state = g.state and p.alt_name = g.key
      where not exists (select 1 from po_all x where x.state = g.state and x.name = g.key))
  select * from h order by place, state, year_established''')
geo = cand[cand.lat.notna()].merge(g[['place', 'state', 'years']], on=['place', 'state'])
geo['open_years'] = [sum(r.year_established <= y and (pd.isna(r.year_discontinued) or r.year_discontinued >= y) for y in r.years)
                     for _, r in geo.iterrows()]
best = (geo.sort_values(['place', 'state', 'open_years', 'year_established'], ascending=[True, True, False, True])
           .drop_duplicates(['place', 'state']))
print('hits:', len(cand), '| geocoded hits:', len(geo), '| places with several geocoded hits:',
      int((geo.groupby(['place', 'state']).size() > 1).sum()), '| matched places:', len(best), 'of', len(g))
display(geo[geo.duplicated(['place', 'state'], keep=False)][['place', 'state', 'name', 'county', 'year_established', 'year_discontinued', 'open_years', 'gnis_class', 'lat', 'lon']])

hits: 112 | geocoded hits: 107 | places with several geocoded hits: 7 | matched places: 99 of 111


,place,state,name,county,year_established,year_discontinued,open_years,gnis_class,lat,lon
2,Allegheny,PA,ALLEGHENY,POTTER,1833,1836,0,Civil,41.864839,-77.904314
3,Allegheny,PA,ALLEGHENY,ALLEGHENY,1836,1911,1,Post Office,40.449917,-80.005139
25,Concord,NH,CONCORD,MERRIMACK,1791,<NA>,1,Post Office,43.207882,-71.526172
26,Concord,NH,CONCORD,GRAFTON,1815,1824,0,Populated Place,44.213396,-71.910923
27,Danvers,MA,DANVERS,ESSEX,1831,1855,1,Post Office,42.565840,-70.934524
28,Danvers,MA,DANVERS,ESSEX,1855,<NA>,0,Post Office,42.565840,-70.934524
71,Norwich,CT,NORWICH,NEW LONDON,1772,1836,3,Post Office,41.524491,-72.071339
72,Norwich,CT,NORWICH,NEW LONDON,1836,<NA>,1,Post Office,41.524491,-72.071339
84,Rochester,NY,ROCHESTER,MONROE,1812,<NA>,2,Post Office,43.084994,-77.595934
85,Rochester,NY,ROCHESTER,ULSTER,1820,1826,0,School,41.793611,-74.224167


In [7]:
u = g.merge(best[['place', 'state', 'id', 'matched_on', 'name', 'county', 'lat', 'lon', 'gnis_dist', 'gnis_class', 'year_established']]
              .rename(columns={'id': 'po_id', 'name': 'po_name', 'county': 'po_county', 'year_established': 'po_established'}),
            on=['place', 'state'], how='left')
unmatched = u[u.lat.isna()].copy()
# hints for hand coordinates only (not used in the join): any exact-name hit in the state, or the same name in another state
unmatched['hint'] = [
    '; '.join(sorted({f"{r2.name}, {r2.state} ({r2.county}) {'no coords' if pd.isna(r2.lat) else 'geocoded'}"
                      for r2 in cand[(cand.place == r.place) & (cand.state == r.state)].itertuples()}))
    or '; '.join(q(f"select distinct name || ', ' || state || ' (' || county || ')' h from po_all where name = '{r.key}' and lat is not null and state <> '{r.state}' and year_established <= {YEAR_MAX}").h)
    or 'no exact name in the file'
    for r in unmatched.itertuples()]
print(f'unmatched urban places ({len(unmatched)}): need hand coordinates')
display(unmatched[['place', 'state', 'year_established', 'hint']])
weak = u[u.lat.notna() & ((u.gnis_class.isin(['Post Office', 'Populated Place']) == False) | (u.gnis_dist.fillna(0) < 1) | (u.matched_on == 'AltName') | (u.po_established > YEAR_MAX))]
print('matched, but check (GNIS class not a post office or populated place, GNIS.Dist < 1 or missing, AltName match, or office established after 1840):')
display(weak[['place', 'state', 'po_name', 'po_county', 'matched_on', 'po_established', 'gnis_class', 'gnis_dist', 'lat', 'lon']])

unmatched urban places (12): need hand coordinates


,place,state,year_established,hint
18,Beverly,MA,1810,"BEVERLY, IL (ADAMS); BEVERLY, WV (RANDOLPH); B..."
25,Gloucester,MA,1790,no exact name in the file
46,Bath,ME,1840,"BATH, ME (LINCOLN) no coords"
50,St. Louis,MO,1830,no exact name in the file
87,Moyamensing,PA,1830,no exact name in the file
88,Northern Liberties,PA,1790,no exact name in the file
93,Southwark,PA,1790,no exact name in the file
96,Cumberland,RI,1840,"CUMBERLAND, OH (GUERNSEY); CUMBERLAND, ME (CUM..."
105,Lynchburg,VA,1830,"LYNCHBURG, VA (CAMPBELL) no coords"
107,Petersburg,VA,1790,"PETERSBURG, VA (DINWIDDIE) no coords"


matched, but check (GNIS class not a post office or populated place, GNIS.Dist < 1 or missing, AltName match, or office established after 1840):


,place,state,po_name,po_county,matched_on,po_established,gnis_class,gnis_dist,lat,lon
30,Middleborough,MA,MIDDLEBORO,PLYMOUTH,AltName,1811.0,Post Office,1.0,41.892797,-70.910782
60,Elizabeth,NJ,ELIZABETH,UNION,Name,1855.0,Post Office,1.0,40.669673,-74.216513
90,Pittsburgh,PA,PITTSBURG,ALLEGHENY,AltName,1788.0,Post Office,1.0,40.458187,-80.019886
104,Fredericksburg,VA,FREDERICKSBURG,SPOTSYLVANIA,Name,1775.0,School,NaN,38.255370,-77.492890
109,Richmond,VA,RICHMOND,HENRICO,Name,1782.0,Locale,NaN,37.512650,-77.355260


### 2b. Hand coordinates (overrides)
`research/urban_coords_override.csv` (R9 §5): USGS GNIS Populated Place points for the 12 unmatched places, and for Richmond and Fredericksburg VA, whose matches were a GNIS *Locale* and a *School*. An override **wins over** the exact-name match. `coord_source` = `post_office` or `override`; `coord_ref` = the point's source. Moyamensing and Southwark have no GNIS feature; their points are landmarks inside the district and are flagged `unverified`.

In [8]:
ov = pd.read_csv(OVERRIDE_CSV)
assert not ov.duplicated(['place', 'state']).any() and ov[['lat', 'lon']].notna().all().all()
assert ov.merge(g[['place', 'state']], how='left', indicator=True)._merge.eq('both').all()   # every override is a Gibson place
uc = u.merge(ov[['place', 'state', 'lat', 'lon', 'source']].rename(columns={'lat': 'ov_lat', 'lon': 'ov_lon'}), on=['place', 'state'], how='left')
is_ov = uc.ov_lat.notna()
uc['coord_source'] = np.where(is_ov, 'override', np.where(uc.lat.notna(), 'post_office', None))
uc['coord_ref'] = np.where(is_ov, uc.source, 'Blevins and Helbock 2021, office po_' + uc.po_id.astype('Int64').astype(str))
uc['unverified'] = [(p, s) in UNVERIFIED for p, s in zip(uc.place, uc.state)]
display(uc[is_ov][['place', 'state', 'lat', 'lon', 'ov_lat', 'ov_lon', 'unverified', 'source']])
uc.loc[is_ov, ['lat', 'lon']] = uc.loc[is_ov, ['ov_lat', 'ov_lon']].values
uc.loc[is_ov, ['po_id', 'gnis_dist', 'gnis_class']] = np.nan            # the office no longer gives the point
assert set(unmatched.place + ', ' + unmatched.state) <= set(ov.place + ', ' + ov.state)   # every unmatched place has hand coordinates
assert uc.lat.notna().all(), uc[uc.lat.isna()][['place', 'state']]
print('coord_source:', uc.coord_source.value_counts().to_dict(), '| unverified:', uc[uc.unverified].place.tolist())


,place,state,lat,lon,ov_lat,ov_lon,unverified,source
18,Beverly,MA,NaN,NaN,42.5584,-70.8800,False,USGS GNIS 614200 (Populated Place)
25,Gloucester,MA,NaN,NaN,42.6159,-70.6620,False,USGS GNIS 615084 (Populated Place)
46,Bath,ME,NaN,NaN,43.9106,-69.8206,False,USGS GNIS 561723 (Populated Place)
50,St. Louis,MO,NaN,NaN,38.6273,-90.1979,False,USGS GNIS 765765 (Populated Place; GNIS name S...
87,Moyamensing,PA,NaN,NaN,39.9319,-75.1612,True,Moyamensing Prison site (Passyunk Ave and Reed...
88,Northern Liberties,PA,NaN,NaN,39.9660,-75.1463,False,USGS GNIS 1202854 (Populated Place)
93,Southwark,PA,NaN,NaN,39.9345,-75.1435,True,"Gloria Dei (Old Swedes') Church, NRHP 66000682..."
96,Cumberland,RI,NaN,NaN,41.9668,-71.4328,False,USGS GNIS 1219346 (Populated Place)
104,Fredericksburg,VA,38.25537,-77.49289,38.3032,-77.4605,False,USGS GNIS 1494947 (Populated Place)
105,Lynchburg,VA,NaN,NaN,37.4138,-79.1422,False,USGS GNIS 1479007 (Populated Place)


coord_source: {'post_office': 97, 'override': 14} | unverified: ['Moyamensing', 'Southwark']


## 3. Write `settlements.json`
Shared fields: `id`, `kind`, `name`, `state`, `lat`, `lon` (4 decimals), `year_established`, `year_discontinued`, `gnis_dist`.
Post-office `name` is as in the source (upper case). Urban rows add `pop` (census year → population, listed years only), `carried_1840`, `place_type`, `alt_name`, `po_id` (the office that gave the coordinates) `note` (merges), `coord_source`, `coord_ref` and `unverified`. Any urban place still without coordinates would be left out and listed in `meta.unmatched_urban` (none after §2b).

In [9]:
def nn(v): return None if pd.isna(v) else v
po_df = q('select * from po order by id')
rows = [dict(id=f'po_{r.id}', kind='post_office', name=r.name, state=r.state, lat=round(r.lat, 4), lon=round(r.lon, 4),
             year_established=int(r.year_established), year_discontinued=nn(r.year_discontinued) and int(r.year_discontinued),
             gnis_dist=nn(r.gnis_dist) and round(float(r.gnis_dist), 3)) for r in po_df.itertuples()]
um = uc
for r in um.itertuples():
    rows.append(dict(id=f"urb_{r.state.lower()}_{re.sub('[^a-z]+', '', r.place.lower())}", kind='urban_2500', name=r.place, state=r.state,
                     lat=round(r.lat, 4), lon=round(r.lon, 4), year_established=int(r.year_established), year_discontinued=None,
                     gnis_dist=nn(r.gnis_dist) and round(float(r.gnis_dist), 3),
                     coord_source=r.coord_source, coord_ref=r.coord_ref, unverified=bool(r.unverified),
                     pop={str(y): int(getattr(r, f'pop_{y}')) for y in CENSUS if pd.notna(getattr(r, f'pop_{y}'))},
                     carried_1840=bool(r.carried_1840), place_type=r.place_type, alt_name=nn(r.alt_name), po_id=nn(r.po_id) and f'po_{int(r.po_id)}',
                     note=nn(r.merge_note)))
meta = {
  'layer': 'Settlement and routes (D-026): dated settled places',
  'grain': 'post_office = one post-office record; urban_2500 = one Gibson urban place (place + state)',
  'fields': {
    'year_established': 'post_office: first year of the record (Established); urban_2500: first census year with >= 2,500 people (a proxy, not a founding date)',
    'year_discontinued': 'post_office: Discontinued (null = open past the data); a rename also closes a record. urban_2500: null',
    'gnis_dist': 'GNIS name-match score of the post office (0.75-1; null = not given). Town-level point, office moves not recorded',
    'pop': 'urban_2500: population by census year (Gibson 1998); years not listed are absent',
    'carried_1840': 'urban_2500: urban in 1830, absent from the 1840 top-100 list (cut at 4,226), kept as urban in 1840 with no population',
    'po_id': 'urban_2500: post-office record that gave the coordinates (exact name + state match); null where coord_source = override',
    'coord_source': 'urban_2500: post_office = point of the exactly matched office; override = hand coordinates from research/urban_coords_override.csv (R9 section 5), which win over the match',
    'coord_ref': 'urban_2500: source of the point (GNIS feature id or landmark) for overrides; the office record for matches',
    'unverified': 'urban_2500: true where the hand point is a landmark, not a GNIS feature (Moyamensing, Southwark)'},
  'caveats': ['Only geocoded offices are kept (about two thirds); short-lived frontier offices are under-represented (Blevins 2021).',
              'Pre-1790 post-office coverage is thin.',
              'Gibson 1840 lists only the top 100 places (>= 4,226).',
              'Moyamensing and Southwark (PA) are landmark points (Moyamensing Prison, Gloria Dei Church), unverified.'],
  'sources': {'post_offices': f'Blevins and Helbock 2021, US Post Offices, Harvard Dataverse V1, doi:10.7910/DVN/NUKCNA, CC0 1.0; file 4491713 read remotely {PO_FETCHED}',
              'urban_places': 'Gibson 1998, Census Population Division Working Paper 27, tables 2-7, public domain (research/urban_places_gibson.csv)',
              'urban_coords_override': 'USGS GNIS geonames map service, Populated Places layer, accessed 2026-11-24; Wikipedia for two landmarks (research/urban_coords_override.csv, R9 section 5)'},
  'extent': EXTENT, 'year_max': YEAR_MAX,
  'counts': {'post_office': len(po_df), 'urban_2500': len(um)},
  'unmatched_urban': [f'{r.place}, {r.state}' for r in uc[uc.lat.isna()].itertuples()],
  'overridden_urban': [f'{r.place}, {r.state}' for r in uc[uc.coord_source == 'override'].itertuples()],
  'notebook': 'rx-pipeline/notebooks/S01_settlements.ipynb', 'task': 'T-016'}
out = WEB / 'settlements.json'
out.write_text(json.dumps({'meta': meta, 'settlements': rows}, separators=(',', ':'), ensure_ascii=False))
print(out.name, f'{out.stat().st_size / 1e6:.2f} MB,', len(rows), 'rows')

settlements.json 2.28 MB, 13580 rows


## 4. Checks

In [10]:
d = json.loads(out.read_text()); R = pd.DataFrame(d['settlements'])
size_mb = out.stat().st_size / 1e6
assert size_mb < 3, size_mb                                                       # T-016 done-when
assert R.id.notna().all() and R.id.is_unique                                      # unique, non-null key
assert len(R) == funnel.inside_extent[0] + len(um) == len(po_df) + len(g)          # every Gibson place drawn
assert R.year_established.notna().all() and R.year_established.between(YEAR_MIN_VALID, YEAR_MAX).all()  # no undated row
assert R.lat.between(S, N).all() and R.lon.between(W, E).all()                    # all inside the extent
assert R[['name', 'state', 'kind']].notna().all().all()
U = R[R.kind == 'urban_2500']
ids_all = set('po_' + q('select id::varchar i from po_all').i)
assert len(U) == len(g) and set(U.name + ', ' + U.state) == set(g.place + ', ' + g.state)
assert U.coord_source.isin(['post_office', 'override']).all()
assert U[U.coord_source == 'post_office'].po_id.isin(ids_all).all()               # matched coordinates come from an office in the file
assert U[U.coord_source == 'override'].po_id.isna().all()
assert set(U[U.coord_source == 'override'].name + ', ' + U[U.coord_source == 'override'].state) == set(ov.place + ', ' + ov.state)  # every override applied
assert set(U[U.unverified].name + ', ' + U[U.unverified].state) == {f'{p}, {s}' for p, s in UNVERIFIED}
print('matched urban places whose office is not a row (established after 1840):', U[(U.coord_source == 'post_office') & ~U.po_id.isin(R.id)].name.tolist())
assert all(str(y) in p for y, p in zip(U.year_established, U['pop']))
assert (U.carried_1840 == U['pop'].map(lambda p: '1830' in p and '1840' not in p)).all()
assert d['meta']['unmatched_urban'] == []
print(f'OK: {len(R)} rows, {size_mb:.2f} MB')

matched urban places whose office is not a row (established after 1840): ['Elizabeth']
OK: 13580 rows, 2.28 MB


In [11]:
# counts by kind x decade of year_established (new places per decade)
R['decade'] = R.year_established // 10 * 10
display(R.pivot_table(index='decade', columns='kind', values='id', aggfunc='count', fill_value=0, margins=True))
# open at each census year: post offices established <= y and not discontinued before y; urban places listed or carried
P = R[R.kind == 'post_office']
active = pd.DataFrame({'post_office': [int(((P.year_established <= y) & ~(P.year_discontinued < y)).sum()) for y in CENSUS],
                       'urban_2500': [int(U['pop'].map(lambda p: str(y) in p).sum() + (U.carried_1840.sum() if y == 1840 else 0)) for y in CENSUS]},
                      index=pd.Index(CENSUS, name='census_year'))
display(active)

kind,post_office,urban_2500,All
decade,,,
1630,1,0,1
1690,1,0,1
1760,1,0,1
1770,59,0,59
1780,39,0,39
1790,566,24,590
1800,1136,9,1145
1810,1774,13,1787
1820,3929,15,3944


,post_office,urban_2500
census_year,,
1790,101,24
1800,821,33
1810,1843,46
1820,3659,61
1830,7166,90
1840,11704,110


## Notes for review
- **Remote read.** The post-office file is read from Dataverse each run (~3 s); nothing is written to `data/raw/` (T-016 change, 2026-11-24).
- **Post offices kept:** `Established` 1600–1840, geocoded, inside the extent. Rows with a 3-digit `Established` (185, 190 ...; truncated years; 17 rows, all IN, KS, OK, TX and the Dakotas, `Discontinued` 1859 or later where given) are dropped as undatable. Three kept offices date before 1770 (Boston 1639, Williamsburg 1699, Falmouth ME 1764). `Discontinued` can be a rename, not a closure.
- **Urban join** is exact name + state only; the 12 unmatched places (§2a) get hand coordinates in §2b. Matches listed as "check" in §2a use a GNIS feature that is not a post office or populated place (e.g. Richmond VA, a *Locale* ~12 km east of the city centre; replaced in §2b), or an office opened after 1840 (Elizabeth NJ, 1855; its `po_id` is not a row in the file). Worth a look.
- **Hand coordinates** (`research/urban_coords_override.csv`, R9 §5) win over the match: 12 unmatched places plus Richmond and Fredericksburg VA. Moyamensing and Southwark are landmark points (`unverified = true`).
- **Philadelphia districts** (Kensington, Spring Garden, Northern Liberties, Southwark, Moyamensing) are parts of the city; the matched ones get their own office's point.
- **Merges:** Sherburne → Nantucket MA; Alexandria VA/DC → Alexandria VA (`note` field).